# v1.1 Strategy Comparison — Baseline vs Avellaneda-Stoikov vs AS+OFI vs Full

Same market-maker code path, same replayed tape, four configurations:

| Label    | use_as | use_ofi | hyst_ticks | use_adaptive_clip |
|----------|--------|---------|------------|-------------------|
| baseline | 0      | 0       | 0          | 0                 |
| as       | 1      | 0       | 0          | 0                 |
| as_ofi   | 1      | 1       | 0          | 0                 |
| full     | 1      | 1       | 1          | 1                 |

All four runs produced by `scripts/run_all_strategies.sh` against the same synthetic tape (or any provided real tape). Each CSV column: `ts_ns, mid, bid, ask, position, real_pnl, unreal_pnl, total_pnl, volume, num_fills, num_requotes, sigma, ofi`.

In [ ]:
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUTDIR  = pathlib.Path('../cmake-build-release/out')
IMG_DIR = pathlib.Path('./img'); IMG_DIR.mkdir(exist_ok=True)

LABELS = ['baseline', 'as', 'as_ofi', 'full']
DISPLAY = {'baseline': 'Baseline (threshold)',
           'as':       'Avellaneda-Stoikov',
           'as_ofi':   'AS + OFI',
           'full':     'AS + OFI + hysteresis + adaptive clip'}

dfs = {}
for lbl in LABELS:
    p = OUTDIR / f'pnl_{lbl}.csv'
    if not p.exists():
        print(f'MISSING: {p} — run scripts/run_all_strategies.sh first')
        continue
    df = pd.read_csv(p)
    df['ts_s'] = df['ts_ns'] / 1e9
    dfs[lbl] = df
    print(f'{lbl:>10}: {len(df):>5} rows, last_pnl={df.total_pnl.iloc[-1]:.2f}, fills={df.num_fills.iloc[-1]}, requotes={df.num_requotes.iloc[-1]}')

## Summary table

In [ ]:
def summarise(df: pd.DataFrame) -> dict:
    last = df.iloc[-1]
    pos = df.position.values
    pnl = df.total_pnl.values
    # 1-second-bin Sharpe approximation (using sample-to-sample pnl delta).
    pnl_diff = np.diff(pnl)
    sharpe = (pnl_diff.mean() / pnl_diff.std()) * np.sqrt(len(pnl_diff)) if pnl_diff.std() > 0 else np.nan
    downside = pnl_diff[pnl_diff < 0]
    sortino = (pnl_diff.mean() / downside.std()) * np.sqrt(len(pnl_diff)) if downside.size and downside.std() > 0 else np.nan
    # Max drawdown over total_pnl.
    high = np.maximum.accumulate(pnl)
    dd   = pnl - high
    return {
        'total_pnl':       last.total_pnl,
        'fills':           int(last.num_fills),
        'requotes':        int(last.num_requotes),
        'fill_rate':       (last.num_fills / last.num_requotes) if last.num_requotes else 0.0,
        'mean_pos':        float(np.mean(pos)),
        '|pos| p95':       float(np.percentile(np.abs(pos), 95)),
        'max_drawdown':    float(dd.min()),
        'sharpe':          float(sharpe),
        'sortino':         float(sortino),
        'final_volume':    int(last.volume),
    }

summary = pd.DataFrame({lbl: summarise(df) for lbl, df in dfs.items()}).T
summary.style.format({'total_pnl':'{:.2f}', 'fill_rate':'{:.3f}',
                      'mean_pos':'{:.2f}', '|pos| p95':'{:.1f}',
                      'max_drawdown':'{:.2f}',
                      'sharpe':'{:.3f}', 'sortino':'{:.3f}'})

## Cumulative PnL

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for lbl, df in dfs.items():
    ax.plot(df.ts_s, df.total_pnl, label=DISPLAY[lbl], lw=1.5)
ax.set_xlabel('time (s)')
ax.set_ylabel('total PnL (ticks)')
ax.set_title('Cumulative PnL — same tape, four strategies')
ax.legend(loc='best')
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(IMG_DIR / 'cumulative_pnl.png', dpi=120)
plt.show()

## Inventory trajectory

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
for lbl, df in dfs.items():
    ax.plot(df.ts_s, df.position, label=DISPLAY[lbl], lw=1.0, alpha=0.85)
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('time (s)')
ax.set_ylabel('inventory (lots)')
ax.set_title('Inventory over time — AS variants pull q toward 0 actively')
ax.legend(loc='best')
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(IMG_DIR / 'inventory.png', dpi=120)
plt.show()

## Inventory histogram

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
all_pos = np.concatenate([df.position.values for df in dfs.values()])
rng = (all_pos.min(), all_pos.max())
for lbl, df in dfs.items():
    ax.hist(df.position, bins=40, range=rng, alpha=0.45, label=DISPLAY[lbl])
ax.set_xlabel('inventory (lots)')
ax.set_ylabel('time spent (samples)')
ax.set_title('Inventory distribution')
ax.legend(loc='best')
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(IMG_DIR / 'inventory_hist.png', dpi=120)
plt.show()

## Drawdown waterfall

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
for lbl, df in dfs.items():
    pnl  = df.total_pnl.values
    high = np.maximum.accumulate(pnl)
    dd   = pnl - high
    ax.plot(df.ts_s, dd, label=DISPLAY[lbl], lw=1.2)
ax.set_xlabel('time (s)')
ax.set_ylabel('drawdown (ticks)')
ax.set_title('Running drawdown — lower is worse')
ax.legend(loc='best')
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(IMG_DIR / 'drawdown.png', dpi=120)
plt.show()

## Feature-engine signals

In [ ]:
if 'full' in dfs:
    df = dfs['full']
    fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
    axes[0].plot(df.ts_s, df.sigma, lw=1.0, color='#9467bd')
    axes[0].set_ylabel('σ (EWMA vol)')
    axes[0].set_title('FeatureEngine signals over the run')
    axes[0].grid(alpha=0.3)
    axes[1].plot(df.ts_s, df.ofi, lw=1.0, color='#2ca02c')
    axes[1].axhline(0, color='k', lw=0.5)
    axes[1].set_xlabel('time (s)')
    axes[1].set_ylabel('OFI (EWMA)')
    axes[1].grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(IMG_DIR / 'features.png', dpi=120)
    plt.show()

## Headline finding

*(Filled in after the runs — compute deltas of full vs baseline on total_pnl, |inventory| p95, Sharpe.)*

In [ ]:
if {'baseline', 'full'}.issubset(dfs):
    b = summarise(dfs['baseline'])
    f = summarise(dfs['full'])
    print(f'Δ total_pnl   :  {b["total_pnl"]:>8.2f}  →  {f["total_pnl"]:>8.2f}    ({(f["total_pnl"]-b["total_pnl"]):+.2f})')
    print(f'Δ |pos| p95   :  {b["|pos| p95"]:>8.1f}  →  {f["|pos| p95"]:>8.1f}    ({(f["|pos| p95"]-b["|pos| p95"]):+.1f})')
    print(f'Δ Sharpe      :  {b["sharpe"]:>8.3f}  →  {f["sharpe"]:>8.3f}    ({(f["sharpe"]-b["sharpe"]):+.3f})')
    print(f'Δ fills       :  {b["fills"]:>8}  →  {f["fills"]:>8}    ({(f["fills"]-b["fills"]):+d})')
    print(f'Δ max_drawdown:  {b["max_drawdown"]:>8.2f}  →  {f["max_drawdown"]:>8.2f}    ({(f["max_drawdown"]-b["max_drawdown"]):+.2f})')